# TF Evolutionary Shift Analysis

This notebook analyzes transcription factor (TF) motif enrichment across evolutionary peak sets using HOMER results. The goal is to identify TFs with conserved or species-specific motif enrichment and to highlight TF rewiring across evolution.

**Analysis steps:**
1. Import required libraries and define file paths
2. Parse knownResults.txt for top TFs in each peak set
3. Compare top TFs across peak sets (shared, human-specific, mouse-specific)
4. Output overlap/difference summary table

---

## 1. Import Required Libraries and Define Paths

Import the necessary Python libraries and define the paths to the HOMER motif result files for each peak set.

In [ ]:
# Import libraries
import pandas as pd
import os

# Define motif result file paths for each peak set
base_dir = "HOMER/results"
peak_sets = {
    "shared": os.path.join(base_dir, "shared_peaks_conservative/motifs/knownResults.txt"),
    "human_specific": os.path.join(base_dir, "human_specific_peaks_conservative/motifs/knownResults.txt"),
    "mouse_specific": os.path.join(base_dir, "mouse_specific_peaks_conservative/motifs/knownResults.txt"),
}

peak_sets

{'shared': 'HOMER/results/shared_peaks_conservative/motifs/knownResults.txt',
 'human_specific': 'HOMER/results/human_specific_peaks_conservative/motifs/knownResults.txt',
 'mouse_specific': 'HOMER/results/mouse_specific_peaks_conservative/motifs/knownResults.txt'}

## 2. Parse knownResults.txt for Top TFs

For each peak set, parse the HOMER motif results and extract the top 20 transcription factors (TFs) based on the most significant p-value or score.

In [ ]:
# Function to debug and parse HOMER knownResults.txt and extract top N TFs

def parse_known_results(filepath, top_n=20):
    print(f"\nFirst 10 lines of {filepath}:")
    with open(filepath, 'r') as f:
        for i in range(10):
            print(f.readline().rstrip())
    # Now try to parse as before
    df = pd.read_csv(filepath, sep='\t', comment='#')
    if 'Motif Name' not in df.columns:
        df = pd.read_csv(filepath, sep='\t', comment='#', skiprows=1)
    motif_col = None
    for col in df.columns:
        if 'motif' in col.lower() and 'name' in col.lower():
            motif_col = col
            break
    if motif_col is None:
        print(f"No motif name column found in {filepath}")
        return []
    df[motif_col] = df[motif_col].astype(str)
    df['TF'] = df[motif_col].str.split('/').str[0]
    top_tfs = df['TF'].head(top_n).tolist()
    return top_tfs

# Parse top 20 TFs for each peak set
top_tfs = {k: parse_known_results(v, 20) for k, v in peak_sets.items()}
top_tfs


Preview of HOMER/results/shared_peaks_conservative/motifs/knownResults.txt:
                                                                             Motif Name  \
AP-1(bZIP)/ThioMac-PU.1-ChIP-Seq(GSE21512)/Homer   VTGACTCATC           1.0         0.0   
AP-2gamma(AP2)/MCF7-TFAP2C-ChIP-Seq(GSE21234)/H... SCCTSAGGSCAW         1.0         0.0   
AP-2alpha(AP2)/Hela-AP2alpha-ChIP-Seq(GSE31477)... ATGCCCTGAGGC         1.0         0.0   
Ap4(bHLH)/AML-Tfap4-ChIP-Seq(GSE45738)/Homer       NAHCAGCTGD           1.0         0.0   
FOXA1:AR(Forkhead,NR)/LNCAP-AR-ChIP-Seq(GSE2782... AGTAAACAAAAAAGAACAND 1.0         0.0   

                                                                             Consensus  \
AP-1(bZIP)/ThioMac-PU.1-ChIP-Seq(GSE21512)/Homer   VTGACTCATC           1.0        1.0   
AP-2gamma(AP2)/MCF7-TFAP2C-ChIP-Seq(GSE21234)/H... SCCTSAGGSCAW         1.0        1.0   
AP-2alpha(AP2)/Hela-AP2alpha-ChIP-Seq(GSE31477)... ATGCCCTGAGGC         1.0        1.0   
Ap4(bHLH)/AML-Tf

{'shared': ['0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0'],
 'human_specific': ['0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0'],
 'mouse_specific': ['0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0',
  '0.0']}

## 3. Compare Top TFs Across Peak Sets

Identify the overlap and differences among the top 20 TFs for shared, human-specific, and mouse-specific peak sets. Output a summary table with columns: TF, shared, human-specific, mouse-specific.

In [ ]:
# Create a summary table of TFs and their presence in each peak set
tf_union = set(top_tfs['shared']) | set(top_tfs['human_specific']) | set(top_tfs['mouse_specific'])
summary = []
for tf in sorted(tf_union):
    summary.append({
        'TF': tf,
        'shared': '✔' if tf in top_tfs['shared'] else '',
        'human-specific': '✔' if tf in top_tfs['human_specific'] else '',
        'mouse-specific': '✔' if tf in top_tfs['mouse_specific'] else ''
    })
tf_summary_df = pd.DataFrame(summary)
tf_summary_df

## 4. Biological Insight: TF Rewiring Across Evolution

The table above highlights TFs that are conserved (shared) or species-specific in motif enrichment. TFs present only in human-specific or mouse-specific peaks may indicate evolutionary rewiring of regulatory networks.